# Clase 131 — Transfer learning con CNNs preentrenadas

Transfer learning en visión: reutilizar una CNN entrenada en ImageNet como
extractor de features y adaptarla a un dataset propio. Receta industrial:
`image_dataset_from_directory` + augmentation, `preprocess_input` del modelo,
entrenamiento en **2 etapas** (head warmup con base congelada → fine-tuning con
LR muy bajo) y el *gotcha* clásico de **BatchNorm** en fine-tuning.

Requiere: `tensorflow` / `keras` (≥ 3.0), `keras.applications`. No se ejecuta
aquí; código idiomático y correcto por API.

## 1. Pipeline de datos con `image_dataset_from_directory`

Carga las imágenes desde un directorio con una subcarpeta por clase e infiere
las etiquetas automáticamente.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
keras.utils.set_random_seed(42)

IMG_SIZE = (224, 224)
NUM_CLASSES = 5

# En la práctica (directorio flores/ con subcarpetas margarita/, rosa/, ...):
# train_ds = keras.utils.image_dataset_from_directory(
#     "flores", validation_split=0.2, subset="training", seed=42,
#     image_size=IMG_SIZE, batch_size=32)
# val_ds = keras.utils.image_dataset_from_directory(
#     "flores", validation_split=0.2, subset="validation", seed=42,
#     image_size=IMG_SIZE, batch_size=32)
print("image_dataset_from_directory infiere las clases de las subcarpetas")

## 2. Augmentation como capas Keras

Las capas de augmentation transforman las imágenes solo durante el training
(Keras pasa `training=True/False` automáticamente).

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")
print("capas de augmentation:", [l.name for l in data_augmentation.layers])

## 3. Modelo = base preentrenada congelada + head nueva

`include_top=False` descarta la cabeza ImageNet; `base.trainable = False`
congela los pesos (feature extraction). Cada modelo trae su `preprocess_input`.

In [ ]:
from tensorflow.keras.applications.xception import preprocess_input

base = keras.applications.Xception(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False   # feature extraction: base congelada

entradas = keras.Input(shape=(224, 224, 3))
x = data_augmentation(entradas)
x = preprocess_input(x)                # escalado específico de Xception
x = base(x, training=False)            # base en modo inference (BN estable)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
salidas = layers.Dense(NUM_CLASSES, activation="softmax")(x)
modelo = keras.Model(entradas, salidas)

trainables = int(sum(np.prod(w.shape) for w in modelo.trainable_weights))
print("params entrenables (base congelada):", trainables)

## 4. Etapa 1: head warmup (base congelada)

Con la base congelada se entrena solo la cabeza, con un LR normal (`1e-3`).

In [ ]:
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])
# history1 = modelo.fit(train_ds, validation_data=val_ds, epochs=5)
print("Etapa 1: head warmup con Adam(1e-3), base congelada")

## 5. Etapa 2: fine-tuning con LR muy bajo

Se descongela la base y se **recompila** (obligatorio tras tocar `trainable`)
con un LR 10–100× menor para no destruir el preentrenamiento.

In [ ]:
base.trainable = True                  # descongelar la base
modelo.compile(optimizer=keras.optimizers.Adam(1e-5),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])
# history2 = modelo.fit(train_ds, validation_data=val_ds, epochs=10)
print("Etapa 2: fine-tuning con Adam(1e-5) (10-100x menor que la etapa 1)")

## 6. El gotcha de BatchNorm en fine-tuning

Al descongelar, las capas `BatchNormalization` seguirían actualizando sus
*moving averages* y podrían dañar el preentrenamiento. En datasets chicos se
mantienen congeladas.

In [ ]:
congeladas = 0
for capa in base.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False
        congeladas += 1
print("capas BatchNormalization mantenidas en inference:", congeladas)
# Alternativa equivalente: pasar training=False a la base (ya hecho en la sec. 3).

## Ejercicios

1. **Augmentation**: construí el `Sequential` de augmentation y listá sus capas.
2. **Base + head**: armá el modelo con `Xception(include_top=False)` congelada,
   `GlobalAveragePooling2D`, `Dropout` y `Dense(NUM_CLASSES)`.
3. **2 etapas**: compilá con `Adam(1e-3)` (etapa 1) y luego, tras
   `base.trainable=True`, recompilá con `Adam(1e-5)` (etapa 2).
4. **BN gotcha**: recorré `base.layers` y congelá las `BatchNormalization`,
   contando cuántas hay.

## Conclusiones

- Transfer learning reutiliza una CNN de ImageNet como extractor de features.
- Cada modelo exige su `preprocess_input`; sin él, el escalado rompe el pretraining.
- La receta de 2 etapas: head warmup (base congelada) → fine-tuning con LR muy bajo.
- Siempre **recompilar** tras cambiar `trainable`, o el optimizer ignora los pesos.
- En datasets chicos, mantené BatchNorm en inference para no dañar el preentrenamiento.